# CN x1.0 — Canonical baseline model

`cn_x1_0` is the first named CN130 research baseline. Status: `trade_ready=false`; provider data review remains open under Issue #345.

## Effective parameter identity correction

The historical calibration schema included LightGBM-oriented fields. For XGBoost, only gain bins and boosting rounds were passed from that schema. The actual `effective_runtime_parameters` are XGBoost `rank:ndcg`, `tree_method=hist`, `grow_policy=lossguide`, `max_leaves=31`, `max_depth=0`, `learning_rate=0.05`, `seed=42`, five gain bins and 100 boosting rounds. `min_data_in_leaf` was not consumed by the XGBoost adapter.

Consequently, the eight-candidate study compared feature groups, gain bins and round counts—not a complete regularization grid. The economic findings and `baseline_only` decision are unchanged.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import yaml

def repo_root(start=Path.cwd()):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / 'pyproject.toml').is_file() and (path / 'configs').is_dir():
            return path
    raise FileNotFoundError('Run inside Alpha Engine')

ROOT = repo_root()
CONFIG = ROOT / 'configs/models/cn_x1_0.yaml'
config = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
assert config['display_name'] == 'CN x1.0'
assert config['trade_ready'] is False
assert config['model']['learning_rate'] == 0.05
assert config['candidate_calibration_identity']['effective_xgb_mapping']['min_data_in_leaf'] == 'not_consumed_by_xgb_adapter'
config['model']

## Contract

- Universe: `cn_selected_equities_v3`, 130 declared equities; static curated and survivorship-biased.
- Benchmark: CSI 300 (`000300`), reference only.
- Canonical provider identity: `bf5fa1373a0b5ebfedcd90c2cf3c4748300efd2b25da0adfbfb1daab8c6405d8`, cutoff 2026-07-31.
- Features: fourteen `cn_balanced_ohlcv` expressions.
- Label/economics: daily cross-sectional gain target; raw 10-session forward return.
- Portfolio: Top-15 equal weight, 10-session holding/rebalance, 20 bps cost.

In [ ]:
pd.DataFrame({'feature_expression': config['features']['expressions']})

In [ ]:
identity = config['candidate_calibration_identity']
pd.DataFrame([
    {'layer': 'declared_identity', 'parameter': 'n_gain_bins', 'value': identity['n_gain_bins']},
    {'layer': 'declared_identity', 'parameter': 'num_boost_round', 'value': identity['num_boost_round']},
    {'layer': 'legacy_not_consumed', 'parameter': 'num_leaves', 'value': identity['legacy_num_leaves_field']},
    {'layer': 'legacy_not_consumed', 'parameter': 'min_data_in_leaf', 'value': identity['legacy_min_data_in_leaf_field']},
    {'layer': 'legacy_not_consumed', 'parameter': 'learning_rate', 'value': identity['legacy_learning_rate_field']},
    *[{'layer': 'effective_runtime_parameters', 'parameter': k, 'value': v} for k, v in config['model'].items() if k not in {'runtime_source', 'calibration_mapping_source', 'parameter_identity_status'}],
])

## Complete backtest evidence

Development covers 2024H1–2025H2. The authoritative `compounded_relative_excess` formula is `(1 + strategy) / (1 + benchmark) - 1`. The 2026H1 evidence was evaluated once in workflow run `30733728747`, artifact `8828889722`, and did not select a new model.

In [ ]:
dev = config['backtest_evidence']['development']
calculated = (1 + dev['compounded_strategy_return']) / (1 + dev['compounded_benchmark_return']) - 1
assert abs(calculated - dev['compounded_relative_excess_return']) < 1e-10
pd.DataFrame(dev['windows'])

In [ ]:
pd.DataFrame([
    {'metric': 'compounded_strategy_return', 'value': dev['compounded_strategy_return']},
    {'metric': 'compounded_benchmark_return', 'value': dev['compounded_benchmark_return']},
    {'metric': 'compounded_relative_excess', 'value': dev['compounded_relative_excess_return']},
    {'metric': 'mean_icir', 'value': dev['mean_icir']},
    {'metric': 'mean_rank_ic', 'value': dev['mean_rank_ic']},
    {'metric': 'mean_top_bottom_spread', 'value': dev['mean_top_bottom_spread']},
    {'metric': 'positive_excess_windows', 'value': dev['positive_excess_windows']},
    {'metric': 'worst_drawdown', 'value': dev['worst_drawdown']},
])

In [ ]:
pd.DataFrame([config['backtest_evidence']['frozen_challenge']]).T.rename(columns={0: '2026H1_value'})

## Provider snapshot sensitivity

The same model contract produced 20.18% compounded relative excess on provider `83f525...779f2` and 13.27% on the canonical provider `bf5fa...c6405d8`. This is not seed drift; it is a data-snapshot governance question. CN x1.0 names the immutable model contract, while performance remains evidence-revision specific.

In [ ]:
comparison = config['backtest_evidence']['snapshot_comparison']
pd.DataFrame([
    {'snapshot': 'prior', **comparison['prior_provider']},
    {'snapshot': 'canonical', **comparison['canonical_provider']},
])

## Interpretation and next version

CN x1.0 is retained because no tested feature/gain-bin/round-count identity displaced it. Mean development Rank IC is only 0.0042, two 2024 windows have negative Rank IC, and 2025H2 underperformed CSI 300. No CN x1.1 factor search is permitted until Issue #345 resolves provider drift and a true XGBoost parameter interface exists.

In [ ]:
VALIDATE = [sys.executable, str(ROOT / 'scripts/validate_model_x1_baselines.py')]
FULL_BACKTEST = ['uv', 'run', 'python', 'scripts/run_cn_feature_quality_validation.py', '--spec', 'configs/research_paradigms/cn_x1_0_frozen_v1.yaml', '--provider-uri', 'artifacts/selected_pool_price_refresh/cn/data/providers/cn', '--output-dir', 'artifacts/evidence/model_versions/cn_x1_0']
print('Contract validation:', ' '.join(VALIDATE))
print('Full backtest:', ' '.join(FULL_BACKTEST))
RUN = False
if RUN:
    subprocess.run(VALIDATE, cwd=ROOT, check=True)